### Overview of products which contain chocolate


In [35]:
import pandas as pd
candy = pd.read_csv('../data/candy-data-cleaned.csv')
candy_chocolate = candy[candy['chocolate'] == 1]

In [36]:
candy_old = pd.read_csv('../data/candy-data.csv')

In [37]:
cat_features = ['fruity', 'caramel', 'peanutyalmondy',
                'nougat', 'crispedricewafer', 'hard', 'bar']
#we leave out pluribus as it is highly associated with bar
candy_space = candy_chocolate[cat_features]
# each observation in candy dataset is a data point, find the furtherst point from all other points

# Calculate the centroid (mean of each binary feature)
centroid = candy_space.mean()

# Find the anti-centroid (flip each bit)
anti_centroid = (centroid < 0.5).astype(int)

# Calculate distances from anti-centroid to all points
from scipy.spatial.distance import cdist
distances_from_anti = cdist([anti_centroid], candy_space, metric='hamming')
mean_distance = distances_from_anti.mean()

print(f"Anti-centroid: {anti_centroid.values}")
print(f"Mean distance to all points: {mean_distance:.3f}")


Anti-centroid: [1 1 1 1 1 1 0]
Mean distance to all points: 0.792


In [38]:
pd.DataFrame({'features': cat_features, 'yes/no': anti_centroid.values})

,features,yes/no
0,fruity,1
1,caramel,1
2,peanutyalmondy,1
3,nougat,1
4,crispedricewafer,1
5,hard,1
6,bar,0


In [39]:
# all these features seem quite crazy
# I am bit concerned about a price, but research shows that customers might be willing to pay more for a unique product
# also "hard" seems strange in connection to all other features
# let's look at other candies which are hard

In [40]:
candy[candy['hard'] == 1]

,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent,mother_company,big_brand
14,Dum Dums,0,1,0,0,0,0,1,0,0,0.732,0.034,0.395,Spangler Candy Company,0
16,Fun Dip,0,1,0,0,0,0,1,0,0,0.732,0.325,0.392,Ferrero SpA,1
17,Gobstopper,0,1,0,0,0,0,1,0,1,0.906,0.453,0.468,Ferrero SpA,1
26,Jawbusters,0,1,0,0,0,0,1,0,1,0.093,0.511,0.281,Ferrero SpA,1
30,Lemonhead,0,1,0,0,0,0,1,0,0,0.046,0.104,0.391,Ferrero SpA,1
41,Nerds,0,1,0,0,0,0,1,0,1,0.848,0.325,0.554,Ferrero SpA,1
49,Pop Rocks,0,1,0,0,0,0,1,0,1,0.604,0.837,0.413,Zeta Espacial S.A.,0
55,Ring pop,0,1,0,0,0,0,1,0,0,0.732,0.965,0.353,Topps,0
57,Root Beer Barrels,0,0,0,0,0,0,1,0,1,0.732,0.069,0.297,Unknown,0
58,Runts,0,1,0,0,0,0,1,0,1,0.872,0.279,0.428,Ferrero SpA,1


In [41]:
# it is a very specific feature and probably not really compatible with other features, it also associates with most of the features negatively, as was shown by matthews correlations

In [42]:
# search the closest candy to our anti-centroid
distances = cdist([anti_centroid], candy_space, metric='hamming')
closest_index = distances.argmin()
closest_candy = candy.iloc[closest_index]
print(f"Closest candy to anti-centroid: {closest_candy['competitorname']}")

Closest candy to anti-centroid: One quarter


In [43]:
closest_candy

competitorname      One quarter
chocolate                     0
fruity                        0
caramel                       0
peanutyalmondy                0
nougat                        0
crispedricewafer              0
hard                          0
bar                           0
pluribus                      0
sugarpercent              0.011
pricepercent              0.511
winpercent                0.461
mother_company          Unknown
big_brand                     0
Name: 3, dtype: object

In [44]:
# find closest candy to our product
product = pd.Series({'chocolate': 1, 'fruity': 1, 'caramel': 1, 'peanutyalmondy': 1,
                      'nougat': 1, 'crispedricewafer': 1, 'hard': 0, 'bar': 1, 'pluribus': 0})
cat_features = ['chocolate', 'fruity', 'caramel', 'peanutyalmondy',
                'nougat', 'crispedricewafer', 'hard', 'bar', 'pluribus']
distances = cdist([product[cat_features]], candy[cat_features], metric='hamming')
closest_index = distances.argmin()
closest_candy = candy.iloc[closest_index]
print(f"Closest candy to our product: {closest_candy['competitorname']}")

Closest candy to our product: Baby Ruth


In [45]:
from catboost import Pool, cv

CAT_FEATURES = ['chocolate', 'fruity', 'caramel', 'peanutyalmondy', 'nougat', 'crispedricewafer', 'hard', 'bar', 'pluribus']

TO_DROP = ['competitorname', 'mother_company', 'big_brand']
X = candy.drop(columns=TO_DROP + ['winpercent'])
y = candy['winpercent']

# Create Pool object with categorical features
pool = Pool(data=X, label=y, cat_features=CAT_FEATURES)

# Parameters WITHOUT cat_features
params = {
    'iterations': 100,
    'learning_rate': 0.05,
    'depth': 6,
    'verbose': 100,
    'loss_function': 'RMSE',
}

# Run cross-validation with Pool object
cv_results = cv(
    pool=pool,
    params=params,
    fold_count=5,
    plot=True,
    return_models=True,
)

print(cv_results)

print(cv_results[1][4].get_feature_importance(prettified=True))

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Training on fold [0/5]
0:	learn: 0.5114632	test: 0.4819019	best: 0.4819019 (0)	total: 56.5ms	remaining: 5.59s
99:	learn: 0.0782373	test: 0.0971015	best: 0.0965046 (74)	total: 202ms	remaining: 0us

bestTest = 0.09650464167
bestIteration = 74

Training on fold [1/5]
0:	learn: 0.4988373	test: 0.5306107	best: 0.5306107 (0)	total: 432us	remaining: 42.8ms
99:	learn: 0.0819399	test: 0.1214659	best: 0.1214659 (99)	total: 117ms	remaining: 0us

bestTest = 0.1214658962
bestIteration = 99

Training on fold [2/5]
0:	learn: 0.5160203	test: 0.4608948	best: 0.4608948 (0)	total: 1.38ms	remaining: 137ms
99:	learn: 0.0725686	test: 0.1109237	best: 0.1109237 (99)	total: 99.4ms	remaining: 0us

bestTest = 0.1109237065
bestIteration = 99

Training on fold [3/5]
0:	learn: 0.5021415	test: 0.5157672	best: 0.5157672 (0)	total: 331us	remaining: 32.8ms
99:	learn: 0.0727380	test: 0.1400872	best: 0.1397335 (94)	total: 171ms	remaining: 0us

bestTest = 0.1397334785
bestIteration = 94

Training on fold [4/5]
0:	learn: 0

In [46]:
# predict the winpercent for our product from trained cv model
# add another feature: sugarpercent and pricepercent
product_full = pd.Series({'chocolate': 1, 'fruity': 1, 'caramel': 1, 'peanutyalmondy': 1,
                      'nougat': 1, 'crispedricewafer': 1, 'hard': 0, 'bar': 1, 'pluribus': 0,
                      'sugarpercent': 0.5, 'pricepercent': 0.5})
for feature in CAT_FEATURES:
    product_full[feature] = int(product_full[feature])
product_df = pd.DataFrame([product_full])
product_df[CAT_FEATURES] = product_df[CAT_FEATURES].astype(int)
predictions = []
for model in cv_results[1]:
    pred = model.predict(product_df)
    predictions.append(pred[0])
average_prediction = sum(predictions) / len(predictions)
print(f"Predicted winpercent for our product: {average_prediction:.2f}%")

Predicted winpercent for our product: 0.51%


In [47]:
candy.winpercent.describe()

count    85.000000
mean      0.503165
std       0.147197
min       0.224000
25%       0.391000
50%       0.478000
75%       0.599000
max       0.842000
Name: winpercent, dtype: float64

In [48]:
#calculate median winpercent
candy.winpercent.median()

np.float64(0.478)

In [49]:
product_df

,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent
0,1,1,1,1,1,1,0,1,0,0.5,0.5
